In [ ]:
"""
El filtrado colaborativo se basa en esta idea:
"Si Juan y María calificaron de forma similar 20 películas, y María vio algo que Juan no vio, probablemente a Juan le guste también."
Para esto se necesita construir una matriz usuario-pelicula
Toy Story  Matrix  Titanic  Heat
usuario 1      5        3       -       4
usuario 2      4        -       5       -
usuario 3      -        5       -       3
Los - son películas que el usuario no vio — y eso es exactamente lo que el modelo intenta predecir.

Para resolverlo vamos a usar una técnica llamada SVD (Singular Value Decomposition) 
— que básicamente comprime la matriz encontrando patrones latentes. En lugar de comparar usuarios directamente, encuentra "perfiles de gusto" en común.
No te preocupes por la matemática detrás por ahora, lo importante es entender qué hace: 
reduce una matriz gigante y dispersa en representaciones más compactas y manejables.

"""

In [13]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

# Cargamos los datos limpios
ratings = pd.read_csv('../data/ratings_clean.csv')
movies = pd.read_csv('../data/movies_clean.csv')

print("Ratings shape:", ratings.shape)
print("Usuarios únicos:", ratings['userId'].nunique())
print("Películas únicas:", ratings['movieId'].nunique())

Ratings shape: (81116, 4)
Usuarios únicos: 610
Películas únicas: 2269


In [14]:
#Construcció nde la matriz usuario - pelicula

# Pivoteamos el dataframe para crear la matriz usuario-película
user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId', 
    values='rating'
).fillna(0)

print("Dimensiones de la matriz:", user_movie_matrix.shape)
user_movie_matrix.head()

Dimensiones de la matriz: (610, 2269)


movieId,1,2,3,5,6,7,9,10,11,12,...,166461,166528,166643,168250,168252,174055,176371,177765,179819,187593
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
# Convertimos a sparse matrix para eficiencia en memoria
matrix_sparse = csr_matrix(user_movie_matrix.values)
print("Tipo:", type(matrix_sparse))
print("Densidad:", matrix_sparse.nnz / (matrix_sparse.shape[0] * matrix_sparse.shape[1]))

Tipo: <class 'scipy.sparse._csr.csr_matrix'>
Densidad: 0.05860601550477209


In [16]:
"""
¿Por qué fillna(0)?
Los algoritmos matemáticos como SVD no pueden operar con NaN — necesitan números. Entonces tenemos que decidir qué significa un valor faltante y representarlo numéricamente.
Usamos 0 porque le decimos al modelo:
"Este usuario no vio esta película — no es que la odió, simplemente no tiene opinión"
Es una convención del dominio: ausencia de rating ≠ rating malo.

EstrategiaQué implicaCuándo usarlafillna(0)
No hay opiniónCuando los datos son implícitos (clicks, vistas)fillna(media_global)
Opinión promedioCuando querés ser conservadorfillna(media_usuario)
Cada usuario tiene su propia "línea base"Más preciso, más complejo

Densidad: 0.05860601550477209 = Solo el 0,058 de la matriz tiene datos reales
"""

'\n¿Por qué fillna(0)?\nLos algoritmos matemáticos como SVD no pueden operar con NaN — necesitan números. Entonces tenemos que decidir qué significa un valor faltante y representarlo numéricamente.\nUsamos 0 porque le decimos al modelo:\n"Este usuario no vio esta película — no es que la odió, simplemente no tiene opinión"\nEs una convención del dominio: ausencia de rating ≠ rating malo.\n\nEstrategiaQué implicaCuándo usarlafillna(0)\nNo hay opiniónCuando los datos son implícitos (clicks, vistas)fillna(media_global)\nOpinión promedioCuando querés ser conservadorfillna(media_usuario)\nCada usuario tiene su propia "línea base"Más preciso, más complejo\n\nDensidad: 0.05860601550477209 = Solo el 0,058 de la matriz tiene datos reales\n'

In [17]:
# n_components = cantidad de "perfiles de gusto" latentes que queremos encontrar
svd = TruncatedSVD(n_components=50, random_state=42)
matrix_svd = svd.fit_transform(matrix_sparse)

print("Matriz original:", matrix_sparse.shape)
print("Matriz comprimida:", matrix_svd.shape)
print(f"Varianza explicada: {svd.explained_variance_ratio_.sum():.2%}")

Matriz original: (610, 2269)
Matriz comprimida: (610, 50)
Varianza explicada: 54.62%


In [18]:
"""
¿Por qué 50 y no otro número?
OpciónProblema5 componentes
Muy pocos perfiles → modelo simplista, pierde información importante
50 componentesBalance razonable → captura patrones sin sobreajustar
500 componentesDemasiados → empieza a memorizar ruido en lugar de patrones reales

Comprimimos de 2.269 columnas a solo 50 — una reducción enorme
Con esas 50 componentes capturamos el 54.62% de la varianza — es decir, más de la mitad de la información relevante del comportamiento de los usuarios
54% puede sonar bajo, pero recordá que el 94% de la matriz era ceros — hay muy poca información real para capturar. Es un resultado esperado y aceptable para este dataset.
"""



'\n¿Por qué 50 y no otro número?\nOpciónProblema5 componentes\nMuy pocos perfiles → modelo simplista, pierde información importante\n50 componentesBalance razonable → captura patrones sin sobreajustar\n500 componentesDemasiados → empieza a memorizar ruido en lugar de patrones reales\n\nComprimimos de 2.269 columnas a solo 50 — una reducción enorme\nCon esas 50 componentes capturamos el 54.62% de la varianza — es decir, más de la mitad de la información relevante del comportamiento de los usuarios\n54% puede sonar bajo, pero recordá que el 94% de la matriz era ceros — hay muy poca información real para capturar. Es un resultado esperado y aceptable para este dataset.\n'

In [19]:
#Que sucede si subimos la cantidad de componentes?

for n in [10, 50, 100, 200, 500]:
    svd_test = TruncatedSVD(n_components=n, random_state=42)
    svd_test.fit_transform(matrix_sparse)
    varianza = svd_test.explained_variance_ratio_.sum()
    print(f"n_components={n:4d} → varianza explicada: {varianza:.2%}")

n_components=  10 → varianza explicada: 34.37%
n_components=  50 → varianza explicada: 54.62%
n_components= 100 → varianza explicada: 69.06%
n_components= 200 → varianza explicada: 85.23%
n_components= 500 → varianza explicada: 99.18%


In [20]:
"""
¿Cuál es el riesgo de usar 500 componentes?
Con 500 componentes y 99% de varianza el modelo empieza a memorizar el ruido de la matriz — los ceros, 
los ratings únicos de usuarios muy particulares — en lugar de aprender patrones generales.
Eso se llama overfitting y el síntoma es:
"El modelo funciona perfecto con los datos que ya conoce, pero falla con usuarios o películas nuevas"
"""

'\n¿Cuál es el riesgo de usar 500 componentes?\nCon 500 componentes y 99% de varianza el modelo empieza a memorizar el ruido de la matriz — los ceros, \nlos ratings únicos de usuarios muy particulares — en lugar de aprender patrones generales.\nEso se llama overfitting y el síntoma es:\n"El modelo funciona perfecto con los datos que ya conoce, pero falla con usuarios o películas nuevas"\n'

In [21]:
#Similitud entre usuarios

from sklearn.metrics.pairwise import cosine_similarity

# Calculamos similitud entre usuarios en el espacio comprimido
user_similarity = cosine_similarity(matrix_svd)

print("Dimensiones:", user_similarity.shape)
print("Similitud usuario 1 con usuario 2:", user_similarity[0][1])

Dimensiones: (610, 610)
Similitud usuario 1 con usuario 2: 0.009794774508439023


In [22]:
"""
Ahora construimos la función de recomendación. Esta es más compleja que la de contenido, así que te explico la lógica antes:
Dado un userId, encontramos los usuarios más similares
Miramos qué películas calificaron bien esos usuarios similares
Filtramos las que el usuario ya vio
Devolvemos las más recomendadas
"""

'\nAhora construimos la función de recomendación. Esta es más compleja que la de contenido, así que te explico la lógica antes:\nDado un userId, encontramos los usuarios más similares\nMiramos qué películas calificaron bien esos usuarios similares\nFiltramos las que el usuario ya vio\nDevolvemos las más recomendadas\n'

In [25]:
#Función de recomendación colaborativa

def get_collaborative_recommendations(user_id, n=10):
    # Paso 1 - Índice del usuario en la matriz
    user_idx = list(user_movie_matrix.index).index(user_id)
    
    # Paso 2 - Usuarios más similares
    sim_scores = list(enumerate(user_similarity[user_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:6]  # Top 5 usuarios similares
    
    # Paso 3 - Películas que vieron esos usuarios
    similar_users_idx = [i[0] for i in sim_scores]
    similar_users_ratings = user_movie_matrix.iloc[similar_users_idx]
    
    # Paso 4 - Películas que el usuario actual NO vio
    user_ratings = user_movie_matrix.iloc[user_idx]
    unseen_movies = user_ratings[user_ratings == 0].index
    
    # Paso 5 - Promedio de ratings de usuarios similares para películas no vistas
    recommendations = similar_users_ratings[unseen_movies].mean(axis=0)
    recommendations = recommendations[recommendations > 0]
    recommendations = recommendations.sort_values(ascending=False).head(n)
    
    # Paso 6 - Unimos con info de películas
    rec_movies = movies[movies['movieId'].isin(recommendations.index)]
    return rec_movies[['title', 'genres']]

In [26]:
get_collaborative_recommendations(user_id=1)

,title,genres
26,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
275,Terminator 2: Judgment Day (1991),Action|Sci-Fi
409,Die Hard (1988),Action|Crime|Thriller
469,Aliens (1986),Action|Adventure|Horror|Sci-Fi
482,Army of Darkness (1993),Action|Adventure|Comedy|Fantasy|Horror
589,Jaws (1975),Action|Horror
644,"Hunt for Red October, The (1990)",Action|Adventure|Thriller
741,"Breakfast Club, The (1985)",Comedy|Drama
826,Blade (1998),Action|Horror|Thriller
1002,"Sixth Sense, The (1999)",Drama|Horror|Mystery


In [28]:
#comparación de ambos modelos
print("=== COLLABORATIVE (Usuario 1) ===")
print(get_collaborative_recommendations(user_id=1))

=== COLLABORATIVE (Usuario 1) ===
                                          title  \
26    Twelve Monkeys (a.k.a. 12 Monkeys) (1995)   
275           Terminator 2: Judgment Day (1991)   
409                             Die Hard (1988)   
469                               Aliens (1986)   
482                     Army of Darkness (1993)   
589                                 Jaws (1975)   
644            Hunt for Red October, The (1990)   
741                  Breakfast Club, The (1985)   
826                                Blade (1998)   
1002                    Sixth Sense, The (1999)   

                                      genres  
26                   Mystery|Sci-Fi|Thriller  
275                            Action|Sci-Fi  
409                    Action|Crime|Thriller  
469           Action|Adventure|Horror|Sci-Fi  
482   Action|Adventure|Comedy|Fantasy|Horror  
589                            Action|Horror  
644                Action|Adventure|Thriller  
741                         